[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/14_bandits_rl_ope/14_proyecto_artwork_bandit_ope.ipynb)

# Proyecto 14 · Bandit contextual de *artwork* para CineMatch, evaluado off-policy

**Nivel:** 🟠 Avanzado / 🔴 Experto · **Duración:** 3–4 h · **Hardware:** CPU (sin GPU) · **Unidades de Colab:** < 1
**Prerrequisitos:** lección 14 (bandits, OPE). Conecta con el módulo 15 (el A/B que vendrá después) y 16 (serving).

## 🎬 Contexto de negocio

Eres ML engineer en el equipo de **Personalización Visual de CineMatch**. Hoy cada título tiene 4 imágenes candidatas (*artwork*) y producción elige una con un modelo antiguo **no personalizado**: un softmax sobre la tasa de *play* histórica de cada imagen, más un 10 % de exploración uniforme. Producción **registra la propensión** de cada impresión (¡bien hecho, equipo de plataforma!).

La directora de producto quiere lanzar un **bandit contextual personalizado**. Antes de gastar tráfico en un A/B (módulo 15), te pide un **dossier de off-policy evaluation**:

1. Demostrar que vuestros estimadores OPE son correctos sobre **datos reales** con propensiones reales: el **Open Bandit Dataset** de ZOZOTOWN (benchmark Random → BTS).
2. Comparar **online en el simulador** de CineMatch varias políticas de exploración (regret).
3. Aprender políticas personalizadas **offline** a partir de los logs de producción y estimar su valor con DM / IPS / SNIPS / DR / Switch-DR, con intervalos de confianza y diagnóstico de ESS.
4. Recomendar **qué política va al A/B** y por qué.

> 💡 El simulador de CineMatch tiene **verdad oculta** (sabemos la probabilidad real de *play*), así que podemos comprobar qué estimador acierta. En la vida real no la tendrías: por eso se calibra la OPE contra A/Bs.

## 📦 Datos

| Dataset | Origen | Uso |
|---|---|---|
| **Open Bandit Dataset** (muestra de 10.000 impresiones por política, incluida en `obp`) | ZOZOTOWN, Saito et al. 2021 (arXiv:2008.07146). Versión completa (~26 M filas): [research.zozo.com/data.html](https://research.zozo.com/data.html), CC BY 4.0 | Validar estimadores con propensiones reales (Random y Bernoulli TS en paralelo) |
| **Simulador de artwork de CineMatch** | Sintético, mismo `ArtworkWorld` de la lección | Políticas online/offline con valor real conocido |

`SCALE = "small"` usa la muestra incluida en `obp`. `SCALE = "full"` usa el OBD completo: descárgalo de la página oficial, descomprímelo en `./open_bandit_dataset/` (o en Drive) y ajusta `OBD_PATH`. Si la ruta no existe, el notebook vuelve automáticamente a la muestra.

## ✅ Entregables y rúbrica

| # | Entregable | Criterio de aceptación |
|---|---|---|
| E1 | `estimate_policy_value` (DM, IPS, SNIPS, DR, Switch-DR, ESS) | Coincide con `obp` en OBD con diferencia absoluta < 1e-8 |
| E2 | Benchmark OBD Random→BTS con IC 95 % bootstrap | Tabla con error relativo vs valor on-policy e IC; discusión de la anchura del IC |
| E3 | Simulación online: producción vs LinUCB vs Thompson lineal | Regret acumulado de **algún** bandit lineal ≤ 25 % del de producción a T=20.000 |
| E4 | Políticas offline (global, personalizada greedy, personalizada softmax) | La mejor tiene **valor real** ≥ +50 % sobre producción |
| E5 | Dossier OPE | Para cada política: 5 estimadores + IC 95 % de DR + ESS. **Error relativo de DR ≤ 5 %** y el IC de DR **cubre** el valor real |
| E6 | Recomendación | Política elegida, riesgos (soporte, ESS, clickbait) y plan de A/B en 5–8 líneas |

In [ ]:
# obp (Open Bandit Pipeline) declara python<3.11 y fija versiones antiguas (torch 1.12,
# sklearn 1.1...). En el Colab actual (Python 3.12) `pip install obp` falla al resolver.
# Su código funciona con numpy/pandas/sklearn/torch modernos, así que lo instalamos SIN
# dependencias (Colab ya trae todas las que usa: numpy, scipy, sklearn, pandas, torch,
# matplotlib, seaborn, tqdm, pyyaml). Probado con obp 0.5.7 (última versión publicada).
!pip install -q --no-deps --ignore-requires-python obp==0.5.7

In [ ]:
import warnings, time, logging
from dataclasses import dataclass
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore"); logging.getLogger("obp").setLevel(logging.WARNING)
seed = 42; rng = np.random.default_rng(seed)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

SCALE = "small"                          # "small" (muestra incluida en obp) | "full" (OBD completo)
OBD_PATH = Path("open_bandit_dataset")   # carpeta descomprimida del OBD completo (solo SCALE="full")
T_ONLINE = 20_000                        # impresiones de la simulación online
N_LOGS_TRAIN, N_LOGS_TEST = 50_000, 25_000

def softmax(z, temp=1.0, axis=-1):
    z = z / temp; z = z - z.max(axis=axis, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def todo_guard(fn, *args, **kw):
    """Ejecuta fn y, si aún es un TODO, avisa sin romper la ejecución del notebook."""
    try:
        return fn(*args, **kw)
    except NotImplementedError:
        print(f"⏳ {fn.__name__}: TODO pendiente (la solución de referencia está al final)")
        return None

### El simulador de artwork de CineMatch (idéntico a la lección)

In [ ]:
THEMES = ["romance", "comedia", "acción", "estrella"]   # "temas" visuales de una imagen

@dataclass
class ArtworkWorld:
    """Mundo sintético de CineMatch: por cada (usuario, título) elegimos 1 de K imágenes.

    - Cada usuario tiene un vector de gustos u ∈ R^4 sobre los temas visuales.
    - Cada imagen k de un título tiene un vector de temas φ_k ∈ {0,1}^4.
    - P(play | u, título, k) = σ(b_título + u·W·φ_k)  ← verdad oculta.
    """
    n_titles: int = 30
    n_art: int = 4
    d_user: int = 6
    seed: int = 0

    def __post_init__(self):
        g = np.random.default_rng(self.seed)
        self.title_bias = g.normal(-3.0, 0.4, self.n_titles)
        # imágenes: cada una resalta 1-2 temas
        self.art = np.zeros((self.n_titles, self.n_art, 4))
        for t in range(self.n_titles):
            for k in range(self.n_art):
                self.art[t, k, k % 4] = 1.0
                if g.random() < 0.3:
                    self.art[t, k, g.integers(4)] = 1.0
        self.W = g.normal(0, 0.9, (self.d_user, 4))    # cómo los rasgos de usuario activan temas
        self.art_quality = g.normal(0, 0.25, (self.n_titles, self.n_art))

    def sample_users(self, n, g):
        """Rasgos de usuario: segmento de gusto dominante + ruido (one-hot suave + continuo)."""
        seg = g.integers(0, 4, n)
        X = g.normal(0, 0.5, (n, self.d_user))
        X[np.arange(n), seg] += 1.5
        return X, seg

    def logits(self, X, t):
        """Logit verdadero para cada imagen del título t. X: (n,d) -> (n,K)."""
        return (self.title_bias[t][:, None] + self.art_quality[t]
                + np.einsum("nd,dj,nkj->nk", X, self.W, self.art[t]))

    def p_play(self, X, t):
        return 1.0 / (1.0 + np.exp(-self.logits(X, t)))

    def features(self, X, t):
        """Features (usuario ⊗ imagen) para un bandit lineal con parámetros compartidos.
        Devuelve (n, K, d_user*4 + 4): producto exterior + temas de la imagen (sesgo por tema)."""
        A = self.art[t]                                   # (n,K,4)
        outer = np.einsum("nd,nkj->nkdj", X, A).reshape(len(X), self.n_art, -1)
        return np.concatenate([outer, A], axis=2)

---
## Parte 1 · Validar los estimadores sobre datos reales (Open Bandit Dataset)

Formato de obp (el que usarás también en el simulador):
- `action_dist`: array `(n, K, L)` con $\pi_e(a\mid x_i, \text{posición } \ell)$; en el simulador $L=1$.
- `q_hat`: array `(n, K, L)` con $\hat q(x_i, a, \ell)$.
- `position`: posición de cada impresión (ceros si $L=1$).

In [ ]:
from obp.dataset import OpenBanditDataset

def load_obd(behavior_policy: str):
    if SCALE == "full" and OBD_PATH.exists():
        return OpenBanditDataset(behavior_policy=behavior_policy, campaign="all", data_path=OBD_PATH)
    if SCALE == "full":
        print(f"⚠️ {OBD_PATH} no existe: uso la muestra pequeña incluida en obp.")
    return OpenBanditDataset(behavior_policy=behavior_policy, campaign="all")

ds_random, ds_bts = load_obd("random"), load_obd("bts")
bf_random = ds_random.obtain_batch_bandit_feedback()
V_BTS_ON_POLICY = OpenBanditDataset.calc_on_policy_policy_value_estimate(
    behavior_policy="bts", campaign="all", data_path=OBD_PATH if (SCALE == "full" and OBD_PATH.exists()) else None)
print(f"Random: n={bf_random['n_rounds']:,}, clics={int(bf_random['reward'].sum())}, K={ds_random.n_actions}, L={ds_random.len_list}")
print(f"Valor on-policy de BTS (verdad del benchmark): {V_BTS_ON_POLICY:.5f}")

### TODO 1 · Implementa los estimadores

Implementa `estimate_policy_value` siguiendo las fórmulas de la lección, **por posición** (como obp):
- $w_i = \pi_e(a_i\mid x_i,\ell_i) / p_i$
- DM: $\frac1n\sum_i\sum_a \pi_e(a\mid x_i,\ell_i)\,\hat q(x_i,a,\ell_i)$
- IPS, SNIPS, DR como en la lección; Switch-DR: término de corrección solo si $w_i \le \tau$.
- ESS $= (\sum w)^2/\sum w^2$.

Pista: indexa con `idx = np.arange(n)` y `action_dist[idx, :, position]` → `(n, K)`.

In [ ]:
def estimate_policy_value(reward, action, pscore, action_dist, q_hat, position=None, tau=50.0) -> dict:
    """Devuelve {'dm','ipw','snipw','dr','switch-dr','ess'} para la política action_dist."""
    # TODO: implementa los cinco estimadores y el ESS
    raise NotImplementedError

def per_round_dr(reward, action, pscore, action_dist, q_hat, position=None) -> np.ndarray:
    """Contribución de cada ronda al estimador DR (útil para el bootstrap)."""
    # TODO
    raise NotImplementedError

In [ ]:
# Política a evaluar: Bernoulli TS de producción de ZOZOTOWN y modelo de recompensa con cross-fitting
from obp.policy import BernoulliTS
from obp.ope import (OffPolicyEvaluation, RegressionModel, DirectMethod, InverseProbabilityWeighting,
                     SelfNormalizedInverseProbabilityWeighting, DoublyRobust, SwitchDoublyRobust)
bts = BernoulliTS(n_actions=ds_random.n_actions, len_list=ds_random.len_list, is_zozotown_prior=True,
                  campaign="all", random_state=seed)
ad_bts = bts.compute_batch_action_dist(n_sim=100_000, n_rounds=bf_random["n_rounds"])
reg = RegressionModel(n_actions=ds_random.n_actions, len_list=ds_random.len_list, action_context=ds_random.action_context,
                      base_model=LogisticRegression(C=100, max_iter=1000, random_state=seed))
qhat_obd = reg.fit_predict(context=bf_random["context"], action=bf_random["action"], reward=bf_random["reward"],
                           position=bf_random["position"], pscore=bf_random["pscore"], n_folds=3, random_state=seed)
ope_obp = OffPolicyEvaluation(bandit_feedback=bf_random, ope_estimators=[
    DirectMethod(), InverseProbabilityWeighting(), SelfNormalizedInverseProbabilityWeighting(),
    DoublyRobust(), SwitchDoublyRobust(lambda_=50.0)])
obp_point = ope_obp.estimate_policy_values(action_dist=ad_bts, estimated_rewards_by_reg_model=qhat_obd)
print({k: round(v, 6) for k, v in obp_point.items()})

In [ ]:
def check_E1():
    mine = estimate_policy_value(bf_random["reward"], bf_random["action"], bf_random["pscore"], ad_bts, qhat_obd,
                                 position=bf_random["position"], tau=50.0)
    diffs = {k: abs(mine[k] - obp_point[k]) for k in obp_point}
    print("Diferencias con obp:", {k: f"{v:.1e}" for k, v in diffs.items()})
    print("E1 ✅" if max(diffs.values()) < 1e-8 else "E1 ❌")
    return mine
_ = todo_guard(check_E1)

### TODO 2 · Bootstrap y benchmark Random → BTS

Implementa `bootstrap_ci(values, n_boot, alpha)` (percentil, remuestreo de rondas) y construye la tabla del benchmark: estimación, IC 95 % (para DR), error relativo $|\hat V - V_{\text{on}}| / V_{\text{on}}$.

In [ ]:
def bootstrap_ci(values: np.ndarray, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0) -> tuple:
    """IC percentil de la media de `values` remuestreando filas con reemplazo."""
    # TODO
    raise NotImplementedError

def obd_benchmark() -> pd.DataFrame:
    # TODO: usa estimate_policy_value, per_round_dr y bootstrap_ci; devuelve un DataFrame por estimador
    raise NotImplementedError
_ = todo_guard(obd_benchmark)

---
## Parte 2 · El mundo CineMatch: política de producción y logs

La política de producción $\pi_0$ ignora al usuario: softmax (temperatura 0,3) sobre una estimación ruidosa de la calidad global de cada imagen, mezclada con un 10 % de exploración uniforme. **Esta celda está completa**: es la "realidad" que te da el equipo de plataforma.

In [ ]:
world = ArtworkWorld(seed=2024)
K = world.n_art
_g = np.random.default_rng(0)
OLD_SCORE = world.title_bias[:, None] + world.art_quality + _g.normal(0, 0.3, (world.n_titles, K))
EPS_PROD = 0.10

def pi_production(X, t):
    """π0(a|x): (n,K). No usa X (no personaliza)."""
    return (1 - EPS_PROD) * softmax(OLD_SCORE[t], temp=0.3) + EPS_PROD / K

def generate_logs(n, g):
    X, seg = world.sample_users(n, g); t = g.integers(0, world.n_titles, n)
    P0 = pi_production(X, t); a = (P0.cumsum(1) > g.random((n, 1))).argmax(1)
    p = world.p_play(X, t)                                       # verdad oculta (solo para evaluar)
    r = (g.random(n) < p[np.arange(n), a]).astype(float)
    return dict(X=X, seg=seg, t=t, a=a, r=r, pscore=P0[np.arange(n), a], P0=P0, p_true=p)

g_logs = np.random.default_rng(1)
logs_train, logs_test = generate_logs(N_LOGS_TRAIN, g_logs), generate_logs(N_LOGS_TEST, g_logs)
true_value = lambda pi, logs: float((pi * logs["p_true"]).sum(1).mean())   # oráculo del simulador
V0 = true_value(logs_test["P0"], logs_test)
print(f"Take rate de producción (real): {V0:.4f} | observado en logs: {logs_test['r'].mean():.4f}")
print(f"Propensión mínima registrada: {logs_train['pscore'].min():.3f}  (⇒ w máximo ≤ {1/logs_train['pscore'].min():.0f})")

---
## Parte 3 · Simulación online: ¿cuánto gana explorar con contexto?

### TODO 3 · Bandits lineales online
Implementa `LinearBandit` (LinUCB y Thompson lineal con Sherman–Morrison) y `simulate_online(policy_name, T)` que devuelva el **regret instantáneo** de cada impresión. Usa `world.features(x, t)` como $x_{t,a}$ (dimensión 28). Políticas: `"production"` (muestrea de $\pi_0$), `"linucb"`, `"lints"`.

In [ ]:
class LinearBandit:
    def __init__(self, d, mode="ucb", alpha=0.5, v=0.3, lam=1.0, rng=None):
        # TODO: guarda A^{-1} (identidad/λ) y b (ceros)
        raise NotImplementedError
    def select(self, Xa):
        # TODO: LinUCB → argmax θ̂ᵀx + α·sqrt(xᵀA⁻¹x);  TS → muestrea θ̃ ~ N(θ̂, v²A⁻¹)
        raise NotImplementedError
    def update(self, x, r):
        # TODO: Sherman–Morrison sobre A^{-1} y b += r x
        raise NotImplementedError

def simulate_online(policy_name: str, T: int, seed: int = 0) -> np.ndarray:
    # TODO: bucle de T impresiones; devuelve regret instantáneo p_max - p[a]
    raise NotImplementedError

online = {name: todo_guard(simulate_online, name, T_ONLINE) for name in ["production", "linucb", "lints"]}

---
## Parte 4 · Aprender políticas offline a partir de los logs

### TODO 4 · Tres políticas candidatas (todas devuelven `(n, K)` probabilidades)
1. `pi_global`: para cada título, la imagen con mayor valor IPS estimado en `logs_train` (determinista, no personaliza).
2. `pi_personal_greedy`: regresión logística sobre `world.features` de la acción mostrada → argmax.
3. `pi_personal_soft`: mismo modelo, `0.95·softmax(score/0.2) + 0.05/K` (estocástica: mejor soporte para OPE y para seguir explorando).

Pista: las features de la acción logueada son `world.features(X, t)[np.arange(n), a]`.

In [ ]:
def fit_reward_model(logs) -> LogisticRegression:
    # TODO
    raise NotImplementedError

def make_policies(logs_train, logs_eval) -> dict:
    """Devuelve {'global': (n,K), 'personal_greedy': (n,K), 'personal_soft': (n,K)} sobre logs_eval."""
    # TODO
    raise NotImplementedError

policies = todo_guard(make_policies, logs_train, logs_test)

---
## Parte 5 · Dossier OPE

### TODO 5 · Estima el valor de cada política en `logs_test`
- Modelo $\hat q$ con **cross-fitting** (2 folds) sobre `logs_test`.
- Para cada política (incluida producción como *sanity check*): DM, IPS, SNIPS, DR, Switch-DR(τ=20), ESS, IC 95 % de DR (bootstrap) y el **valor real** (oráculo) para calificar.
- Comprueba E4 y E5 y dibuja un gráfico de estimaciones con IC vs valor real.

In [ ]:
def crossfit_qhat(logs, n_folds=2, seed=0) -> np.ndarray:
    """q̂(x_i, a) para todas las acciones, (n, K), entrenado fuera del fold de cada fila."""
    # TODO
    raise NotImplementedError

def ope_dossier(logs, policies) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

dossier = todo_guard(ope_dossier, logs_test, policies) if policies is not None else None

### TODO 6 · Recomendación (escribe en esta celda)

*¿Qué política mandas al A/B? ¿Con qué exploración residual? ¿Qué guardrails vigilarías (finalización, tiempo hasta play, quejas de clickbait)? ¿Qué harías si el ESS fuese < 1 % de n?*

---
# ⛔ SPOILER — intenta resolverlo primero

A partir de aquí está la **solución de referencia** completa. Redefine las funciones de los TODOs y ejecuta todo el pipeline con los criterios de la rúbrica.

### Solución · Parte 1 (E1, E2)

In [ ]:
def estimate_policy_value(reward, action, pscore, action_dist, q_hat, position=None, tau=50.0) -> dict:
    n = len(reward); idx = np.arange(n)
    pos = np.zeros(n, dtype=int) if position is None else position
    pi_obs = action_dist[idx, action, pos]
    w = pi_obs / pscore
    pi_all, q_all = action_dist[idx, :, pos], q_hat[idx, :, pos]          # (n, K)
    dm_i = (pi_all * q_all).sum(1)
    resid = reward - q_all[idx, action]
    return {"dm": dm_i.mean(), "ipw": (w * reward).mean(), "snipw": (w * reward).sum() / w.sum(),
            "dr": (dm_i + w * resid).mean(), "switch-dr": (dm_i + (w <= tau) * w * resid).mean(),
            "ess": w.sum() ** 2 / (w ** 2).sum()}

def per_round_dr(reward, action, pscore, action_dist, q_hat, position=None) -> np.ndarray:
    n = len(reward); idx = np.arange(n)
    pos = np.zeros(n, dtype=int) if position is None else position
    w = action_dist[idx, action, pos] / pscore
    q_all = q_hat[idx, :, pos]
    return (action_dist[idx, :, pos] * q_all).sum(1) + w * (reward - q_all[idx, action])

def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=0):
    g = np.random.default_rng(seed); n = len(values)
    means = np.array([values[g.integers(0, n, n)].mean() for _ in range(n_boot)])
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

mine_obd = check_E1()

In [ ]:
def obd_benchmark() -> pd.DataFrame:
    b = bf_random
    est = estimate_policy_value(b["reward"], b["action"], b["pscore"], ad_bts, qhat_obd, b["position"], tau=50.0)
    lo, hi = bootstrap_ci(per_round_dr(b["reward"], b["action"], b["pscore"], ad_bts, qhat_obd, b["position"]))
    rows = [dict(estimador=k, estimación=v, error_relativo=abs(v - V_BTS_ON_POLICY) / V_BTS_ON_POLICY)
            for k, v in est.items() if k != "ess"]
    df = pd.DataFrame(rows).set_index("estimador")
    print(f"IC 95 % de DR: [{lo:.5f}, {hi:.5f}]  (anchura relativa {(hi - lo) / V_BTS_ON_POLICY:.0%})"
          f" | verdad on-policy {V_BTS_ON_POLICY:.5f} | ESS={est['ess']:.0f}")
    return df

display(obd_benchmark().round(5))
print("""Lectura: con la muestra de 10k filas (~40 clics) el IC de DR abarca un rango enorme; las
estimaciones puntuales son orientativas. Con SCALE='full' (1,4 M filas en Random/all) el IC se estrecha
~12× (√(n_full/n_small)). DM sale estrecho pero sesgado: refleja la incertidumbre de un modelo, no la causal.""")

### Solución · Parte 3 (E3)

In [ ]:
class LinearBandit:
    def __init__(self, d, mode="ucb", alpha=0.5, v=0.3, lam=1.0, rng=None):
        self.mode, self.alpha, self.v = mode, alpha, v
        self.A_inv, self.b = np.eye(d) / lam, np.zeros(d)
        self.rng = rng or np.random.default_rng(0)
    def select(self, Xa):
        theta = self.A_inv @ self.b
        if self.mode == "ucb":
            bonus = np.sqrt(np.einsum("kd,de,ke->k", Xa, self.A_inv, Xa))
            return int(np.argmax(Xa @ theta + self.alpha * bonus))
        L = np.linalg.cholesky(self.v ** 2 * self.A_inv + 1e-10 * np.eye(len(self.b)))
        return int(np.argmax(Xa @ (theta + L @ self.rng.standard_normal(len(self.b)))))
    def update(self, x, r):
        Ax = self.A_inv @ x
        self.A_inv -= np.outer(Ax, Ax) / (1.0 + x @ Ax)
        self.b += r * x

def simulate_online(policy_name: str, T: int, seed: int = 0) -> np.ndarray:
    g = np.random.default_rng(seed)
    X, _ = world.sample_users(T, g); titles = g.integers(0, world.n_titles, T)
    d = world.d_user * 4 + 4
    bandit = None if policy_name == "production" else LinearBandit(d, mode="ucb" if policy_name == "linucb" else "ts", rng=g)
    regret = np.empty(T)
    for t in range(T):
        x, ti = X[t:t + 1], titles[t:t + 1]
        p = world.p_play(x, ti)[0]
        if bandit is None:
            a = int((pi_production(x, ti)[0].cumsum() > g.random()).argmax())
        else:
            Xa = world.features(x, ti)[0]; a = bandit.select(Xa)
            bandit.update(Xa[a], float(g.random() < p[a]))
        regret[t] = p.max() - p[a]
    return regret

t0 = time.time()
online = {name: simulate_online(name, T_ONLINE) for name in ["production", "linucb", "lints"]}
print(f"{time.time() - t0:.1f}s")
plt.figure(figsize=(8, 4))
for name, c in zip(online, ["#9e9e9e", "#009e73", "#cc79a7"]):
    plt.plot(np.cumsum(online[name]), label=name, color=c, lw=2)
plt.title("Regret acumulado online en CineMatch"); plt.xlabel("impresión"); plt.ylabel("plays perdidos esperados")
plt.legend(); plt.show()
ratio = min(online["linucb"].sum(), online["lints"].sum()) / online["production"].sum()
print(f"Regret mejor bandit / producción = {ratio:.2f}  →  E3 {'✅' if ratio <= 0.25 else '❌'}")

### Solución · Parte 4 (E4)

In [ ]:
def logged_features(logs):
    F = world.features(logs["X"], logs["t"])
    return F, F[np.arange(len(logs["a"])), logs["a"]]

def fit_reward_model(logs) -> LogisticRegression:
    _, Fa = logged_features(logs)
    return LogisticRegression(C=1.0, max_iter=3000).fit(Fa, logs["r"])

def make_policies(logs_train, logs_eval) -> dict:
    n = len(logs_eval["a"])
    # 1) Global por título con valor IPS por (título, imagen)
    df = pd.DataFrame(dict(t=logs_train["t"], a=logs_train["a"], rw=logs_train["r"] / logs_train["pscore"]))
    ips_ta = (df.groupby(["t", "a"]).rw.sum().unstack().reindex(columns=range(K)).fillna(0)
              / df.groupby("t").size().values[:, None])
    best = ips_ta.values.argmax(1)
    pi_global = np.eye(K)[best[logs_eval["t"]]]
    # 2) y 3) Personalizadas
    clf = fit_reward_model(logs_train)
    F, _ = logged_features(logs_eval)
    S = clf.decision_function(F.reshape(-1, F.shape[-1])).reshape(n, K)
    return {"global": pi_global, "personal_greedy": np.eye(K)[S.argmax(1)],
            "personal_soft": 0.95 * softmax(S, temp=0.2) + 0.05 / K}

policies = make_policies(logs_train, logs_test)
vals = {k: true_value(v, logs_test) for k, v in policies.items()}
print({k: round(v, 4) for k, v in vals.items()}, "| producción:", round(V0, 4))
best_lift = max(vals.values()) / V0 - 1
print(f"Mejor lift real vs producción: {best_lift:+.0%}  →  E4 {'✅' if best_lift >= 0.5 else '❌'}")

### Solución · Parte 5 (E5)

In [ ]:
def crossfit_qhat(logs, n_folds=2, seed=0) -> np.ndarray:
    n = len(logs["a"]); folds = np.random.default_rng(seed).integers(0, n_folds, n)
    F, Fa = logged_features(logs); out = np.zeros((n, K))
    for f in range(n_folds):
        tr_ = folds != f
        clf = LogisticRegression(C=1.0, max_iter=3000).fit(Fa[tr_], logs["r"][tr_])
        out[~tr_] = clf.predict_proba(F[~tr_].reshape(-1, F.shape[-1]))[:, 1].reshape(-1, K)
    return out

def ope_dossier(logs, policies) -> pd.DataFrame:
    qh = crossfit_qhat(logs)[:, :, None]
    rows = []
    for name, pi in {"production": logs["P0"], **policies}.items():
        ad = pi[:, :, None]
        est = estimate_policy_value(logs["r"], logs["a"], logs["pscore"], ad, qh, tau=20.0)
        lo, hi = bootstrap_ci(per_round_dr(logs["r"], logs["a"], logs["pscore"], ad, qh), n_boot=500)
        v = true_value(pi, logs)
        rows.append(dict(política=name, real=v, **{k: est[k] for k in ["dm", "ipw", "snipw", "dr", "switch-dr"]},
                         dr_lo=lo, dr_hi=hi, ess=est["ess"], err_rel_dr=abs(est["dr"] - v) / v,
                         cubre=lo <= v <= hi))
    return pd.DataFrame(rows).set_index("política")

dossier = ope_dossier(logs_test, policies)
display(dossier.round(4))
ok = (dossier.err_rel_dr <= 0.05).all() and dossier.cubre.all()
print("E5", "✅" if ok else "❌ (revisa: ¿algún IC no cubre? Mira el ESS)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
names = dossier.index.tolist(); x = np.arange(len(names)); offs = np.linspace(-0.3, 0.3, 5)
for o, est, c in zip(offs, ["dm", "ipw", "snipw", "dr", "switch-dr"], ["#56b4e9", "#e69f00", "#c9b200", "#009e73", "#cc79a7"]):
    ax.scatter(x + o, dossier[est], color=c, label=est, zorder=3)
ax.errorbar(x + offs[3], dossier.dr, yerr=[dossier.dr - dossier.dr_lo, dossier.dr_hi - dossier.dr],
            fmt="none", ecolor="#009e73", capsize=4)
ax.scatter(x, dossier.real, marker="_", s=900, color="k", label="valor real (oráculo)", zorder=4)
ax.set_xticks(x, names); ax.set(title="Dossier OPE: estimaciones vs valor real (IC 95 % de DR)", ylabel="take rate")
ax.legend(ncol=3, fontsize=8); plt.tight_layout(); plt.show()

### Solución · Recomendación (E6, ejemplo)

> **Recomendación:** llevar `personal_soft` a un A/B 50/50 contra producción. Aunque `personal_greedy` tiene un valor estimado algo mayor, la versión softmax (i) mantiene un 5 % de exploración y propensiones > 0 para todas las imágenes, lo que permite seguir haciendo OPE y re-entrenar sin sesgo; (ii) tiene un ESS mayor, así que su estimación es más fiable; (iii) reduce el riesgo de que el modelo "se case" con una imagen. Guardrails del A/B: tasa de finalización / minutos tras el play (anti-clickbait), tiempo hasta el primer play, abandono de la ficha. Si el ESS fuese < 1 % de n, no decidiríamos con OPE: subiríamos la exploración de producción unas semanas o evaluaríamos solo en el subconjunto de tráfico explorado.

---
## 🚀 Retos extra (nivel experto)

1. **OBD completo** (`SCALE="full"`): repite el benchmark Random→BTS por campaña (*all/men/women*) y compara tu error relativo con la tabla del paper de Saito et al.
2. **Estimadores avanzados en obp:** `DoublyRobustWithShrinkage` (DRos), `SelfNormalizedDoublyRobust` y selección automática de τ (`obp.ope` incluye utilidades de *hyperparameter tuning*). ¿Ganan a tu Switch-DR?
3. **Off-policy learning:** entrena `obp.policy.IPWLearner` o un *neural-linear bandit* en PyTorch (GPU) y compáralo con tu `personal_soft` por DR.
4. **Recompensa anti-clickbait:** modifica `ArtworkWorld` para que algunas imágenes suban el *play* pero bajen la finalización; redefine la recompensa como "play con ≥ 70 % visto" y observa cómo cambia la política elegida.
5. **No estacionariedad:** haz que los gustos cambien a mitad de la simulación online y compara LinUCB con ventana deslizante vs sin ventana.

## 🤔 Reflexión (conexión con producción / MLOps)

- ¿Dónde se registra la propensión en vuestra arquitectura (servicio de decisión, Kafka, feature store)? ¿Qué pasa si una regla de negocio filtra una imagen *después* de la política?
- ¿Cómo versionarías juntos política, modelo de recompensa y datasets de OPE en MLflow para que el dossier sea reproducible?
- ¿Con qué frecuencia re-entrenarías el bandit y cómo monitorizarías el ESS y la deriva de propensiones (módulo 17)?
- ¿Qué métrica del A/B del módulo 15 usarías como *north star* para el artwork y cuáles como guardrails?